# Caso C — Data Augmentation Combinada (Cross-Matching con Perturbación)

## 1. Fundamento y Objetivo
Mientras que el Caso A genera casos bajos (0.0% a 0.40%) al cruzar perfiles ideales, y el Caso B genera casos altos (0.50% a 0.85%) sobre la carrera propia, se requiere una estrategia que densifique y pueble con realismo los **rangos intermedios y de frontera vocacional** (0.25% a 0.75%).

El **Caso C (Estrategia Combinada)**:
1. Toma un usuario Happy Path de un área.
2. Aplica perturbación estocástica a su repertorio de habilidades (reducción de cumplimiento moderada [0.50 a 0.85] y dropout de 1 a 2 habilidades).
3. Cruza ese perfil perturbado contra **todas las carreras del área**.

**Resultado:** Genera un espectro denso, continuo y diversificado que simula a estudiantes con fortalezas heterogéneas evaluándose en múltiples opciones académicas.

### Esquema de Identificación:
Cada usuario generado lleva el prefijo de UUID: `COMBO-<uuid>`.

### Celda 1: Importaciones y Conexión

In [1]:
import random
import uuid
from typing import Optional

import polars as pl
import sqlalchemy as sa
from faker import Faker
from IPython.display import display
from pydantic import BaseModel

DB_HOST = '100.95.220.1'
DB_PORT = 5432
DB_USER = 'admin'
DB_PASSWORD = 'password'
DB_CONECTION = 'athena'

connection_url = (
    f'postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}'
    f'@{DB_HOST}:{DB_PORT}/{DB_CONECTION}'
)
engine = sa.create_engine(connection_url)
fake = Faker('es_MX')

### Celda 2: Modelos Pydantic

In [2]:
# ── Modelos Pydantic (Validación de Esquema) ─────────────────────────
class Area(BaseModel):
    id_area: int
    nombre_area: str

class Carrera(BaseModel):
    id_carrera: int
    nombre_carrera: str

class Habilidad(BaseModel):
    id_habilidad: int
    descripcion: str

class Usuario(BaseModel):
    id_usuario: Optional[int] = None
    uuid_usuario: str
    nombre: str
    correo: str
    id_area: Optional[int] = None

class Resultado(BaseModel):
    id_resultado: Optional[int] = None
    uuid_usuario: str
    id_carrera: int
    porcentaje_coincidencia: float = 1.0
    top: int = 1

### Celda 3: Helpers de Base de Datos

In [3]:
# ── Funciones Helper de Consulta y Persistencia en BD ──────────────────
def get_all_areas() -> list[dict]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_area, nombre_area
            FROM area
            ORDER BY id_area;
        ''')).mappings().all()
    return [
        {'id_area': row['id_area'], 'nombre_area': row['nombre_area']}
        for row in rows
    ]


def get_all_careers() -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            ORDER BY id_carrera;
        ''')).mappings().all()
    return [
        Carrera(
            id_carrera=row['id_carrera'],
            nombre_carrera=row['nombre_carrera']
        )
        for row in rows
    ]


def get_careers_by_area_select(id_area: int) -> list[Carrera]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT id_carrera, nombre_carrera
            FROM carrera
            WHERE id_area = :id_area
            ORDER BY id_carrera;
        '''), {'id_area': id_area}).mappings().all()
    return [
        Carrera(
            id_carrera=row['id_carrera'],
            nombre_carrera=row['nombre_carrera']
        )
        for row in rows
    ]


def get_habilities_by_career(id_carrera: int) -> list[Habilidad]:
    with engine.begin() as conn:
        rows = conn.execute(sa.text('''
            SELECT h.id_habilidad, h.descripcion
            FROM carrera_habilidad ch
            JOIN habilidad h ON h.id_habilidad = ch.id_habilidad
            WHERE ch.id_carrera = :id_carrera
            ORDER BY h.id_habilidad;
        '''), {'id_carrera': id_carrera}).mappings().all()
    return [
        Habilidad(
            id_habilidad=row['id_habilidad'],
            descripcion=row['descripcion']
        )
        for row in rows
    ]


def submit_user(user: Usuario) -> Usuario:
    with engine.begin() as conn:
        row = conn.execute(sa.text('''
            INSERT INTO usuario (nombre, correo, uuid_usuario, id_area)
            VALUES (:nombre, :correo, :uuid_usuario, :id_area)
            RETURNING id_usuario, uuid_usuario;
        '''), {
            'nombre': user.nombre,
            'correo': user.correo,
            'uuid_usuario': user.uuid_usuario,
            'id_area': user.id_area
        }).mappings().one()
    return Usuario(
        id_usuario=row['id_usuario'],
        uuid_usuario=str(row['uuid_usuario']),
        nombre=user.nombre,
        correo=user.correo,
        id_area=user.id_area,
    )


def construct_result(
    uuid_usuario: str,
    id_carrera: int,
    porcentaje_coincidencia: float = 1.0,
    top: int = 1
) -> Resultado:
    with engine.begin() as conn:
        conn.execute(sa.text('''
            INSERT INTO resultado
                (uuid_usuario, id_carrera, porcentaje_coincidencia, top)
            VALUES (:uuid_usuario, :id_carrera, :porcentaje_coincidencia, :top)
            ON CONFLICT (uuid_usuario, id_carrera) DO UPDATE SET
                porcentaje_coincidencia = EXCLUDED.porcentaje_coincidencia,
                top = EXCLUDED.top;
        '''), {
            'uuid_usuario': uuid_usuario,
            'id_carrera': id_carrera,
            'porcentaje_coincidencia': porcentaje_coincidencia,
            'top': top,
        })
    return Resultado(
        uuid_usuario=uuid_usuario,
        id_carrera=id_carrera,
        porcentaje_coincidencia=porcentaje_coincidencia,
        top=top,
    )


def get_happy_path_users_by_area(id_area: int) -> list[dict]:
    '''
    Extrae los usuarios Happy Path originales del area,
    agrupando sus habilidades evaluadas y cumplimiento real.
    '''
    query = '''
        SELECT
            u.id_usuario,
            u.uuid_usuario,
            u.nombre,
            u.correo,
            u.id_area,
            uh.id_habilidad,
            MAX(uh.cumplimiento_criterio) AS cumplimiento_criterio
        FROM usuario u
        JOIN usuario_habilidad uh ON uh.id_usuario = u.id_usuario
        WHERE u.id_area = :id_area
          AND u.uuid_usuario LIKE 'USER-%'
        GROUP BY u.id_usuario, u.uuid_usuario, u.nombre, u.correo,
                 u.id_area, uh.id_habilidad
        ORDER BY u.id_usuario, uh.id_habilidad;
    '''
    with engine.begin() as conn:
        rows = conn.execute(
            sa.text(query),
            {'id_area': id_area}
        ).mappings().all()
    usuarios = {}
    for row in rows:
        uid = row['id_usuario']
        if uid not in usuarios:
            usuarios[uid] = {
                'id_usuario': uid,
                'uuid_usuario': row['uuid_usuario'],
                'nombre': row['nombre'],
                'correo': row['correo'],
                'id_area': row['id_area'],
                'habilidades': {}
            }
        usuarios[uid]['habilidades'][row['id_habilidad']] = float(
            row['cumplimiento_criterio']
        )
    return list(usuarios.values())

In [4]:
def generate_name_and_email() -> tuple[str, str]:
    name = fake.name()
    fake_email = fake.email()
    email_local, email_domain = fake_email.split('@', 1)
    email = f'{email_local}_{uuid.uuid4().hex[:10]}@{email_domain}'
    return name, email


_uncached_get_habilities_by_career = get_habilities_by_career
_career_skills_cache = {}


def get_habilities_by_career(id_carrera: int) -> list[Habilidad]:
    if id_carrera not in _career_skills_cache:
        _career_skills_cache[id_carrera] = _uncached_get_habilities_by_career(id_carrera)
    return _career_skills_cache[id_carrera]

### Celda 4: Algoritmo Combinado (Perturbación + Cruce Multicarrera)

In [5]:
N_VARIACIONES_COMBO = 2
PCTG_REDUCCION_COMBO = 0.30
MIN_CUMPL_COMBO = 0.50
MAX_CUMPL_COMBO = 0.85
MIN_DROPOUT_COMBO = 1
MAX_DROPOUT_COMBO = 2


def persist_cases(pending: list[dict], resumen: list[dict]) -> None:
    if not pending:
        return
    with engine.begin() as conn:
        conn.execute(sa.text('''
            INSERT INTO usuario (nombre, correo, uuid_usuario, id_area)
            VALUES (:nombre, :correo, :uuid_usuario, :id_area);
        '''), [
            {
                'nombre': item['nombre'],
                'correo': item['correo'],
                'uuid_usuario': item['uuid_usuario'],
                'id_area': item['id_area']
            }
            for item in pending
        ])
        inserted = conn.execute(sa.text('''
            SELECT id_usuario, uuid_usuario
            FROM usuario
            WHERE uuid_usuario = ANY(:uuids);
        '''), {'uuids': [item['uuid_usuario'] for item in pending]}).mappings().all()
        ids_by_uuid = {row['uuid_usuario']: row['id_usuario'] for row in inserted}
        for item in pending:
            item['id_usuario'] = ids_by_uuid[item['uuid_usuario']]
            resumen[item['summary_index']]['id_usuario'] = item['id_usuario']

        skills = [
            {
                'id_usuario': item['id_usuario'],
                'id_habilidad': skill_id,
                'cumplimiento_criterio': value
            }
            for item in pending
            for skill_id, value in item['habilidades'].items()
        ]
        conn.execute(sa.text('''
            INSERT INTO usuario_habilidad
                (id_usuario, id_habilidad, cumplimiento_criterio)
            VALUES (:id_usuario, :id_habilidad, :cumplimiento_criterio);
        '''), skills)
        results = [
            {
                'uuid_usuario': item['uuid_usuario'],
                'id_carrera': item['id_carrera'],
                'porcentaje_coincidencia': item['porcentaje_coincidencia'],
                'top': 1
            }
            for item in pending
        ]
        conn.execute(sa.text('''
            INSERT INTO resultado
                (uuid_usuario, id_carrera, porcentaje_coincidencia, top)
            VALUES (:uuid_usuario, :id_carrera, :porcentaje_coincidencia, :top)
            ON CONFLICT (uuid_usuario, id_carrera) DO UPDATE SET
                porcentaje_coincidencia = EXCLUDED.porcentaje_coincidencia,
                top = EXCLUDED.top;
        '''), results)


def run_combined_cross_perturbation(persist_to_db: bool = True) -> list[dict]:
    resumen = []
    pending = []
    career_cache = {}
    for area in get_all_areas():
        id_area = area['id_area']
        careers = get_careers_by_area_select(id_area)
        for usuario_hp in get_happy_path_users_by_area(id_area):
            original = dict(usuario_hp['habilidades'])
            for variation in range(N_VARIACIONES_COMBO):
                varied = dict(original)
                reduce_count = max(1, int(len(varied) * PCTG_REDUCCION_COMBO))
                for skill_id in random.sample(
                    list(varied), k=min(reduce_count, len(varied))
                ):
                    varied[skill_id] = round(
                        random.uniform(MIN_CUMPL_COMBO, MAX_CUMPL_COMBO), 2
                    )
                dropout_count = random.randint(
                    MIN_DROPOUT_COMBO,
                    min(MAX_DROPOUT_COMBO, max(1, len(varied) - 1))
                )
                if len(varied) > 1:
                    for skill_id in random.sample(
                        list(varied), k=dropout_count
                    ):
                        del varied[skill_id]

                for career in careers:
                    career_id = career.id_carrera
                    if career_id not in career_cache:
                        career_cache[career_id] = get_habilities_by_career(career_id)
                    required_ids = {
                        skill.id_habilidad
                        for skill in career_cache[career_id]
                    }
                    total_required = len(required_ids)
                    common_ids = required_ids.intersection(varied)
                    completion_sum = sum(varied[skill_id] for skill_id in common_ids)
                    percentage = (
                        round(completion_sum / total_required, 4)
                        if total_required else 0.0
                    )
                    if percentage >= 1.0:
                        continue

                    generated_uuid = f'COMBO-{uuid.uuid4()}'
                    name, email = generate_name_and_email()
                    summary_index = len(resumen)
                    resumen.append({
                        'id_usuario': None,
                        'uuid_usuario': generated_uuid,
                        'variacion': variation + 1,
                        'id_area': id_area,
                        'id_carrera': career_id,
                        'carrera_evaluada': career.nombre_carrera,
                        'total_requeridas': total_required,
                        'habilidades_perfil_ruidoso': len(varied),
                        'habilidades_comunes': len(common_ids),
                        'suma_cumplimiento': round(completion_sum, 2),
                        'porcentaje_coincidencia': percentage
                    })
                    pending.append({
                        'summary_index': summary_index,
                        'uuid_usuario': generated_uuid,
                        'nombre': name,
                        'correo': email,
                        'id_area': id_area,
                        'id_carrera': career_id,
                        'habilidades': varied,
                        'porcentaje_coincidencia': percentage
                    })

    by_area = {}
    for item in pending:
        by_area.setdefault(item['id_area'], []).append(item)
    if len(by_area) < 3:
        raise ValueError('No hay candidatos en las tres areas.')
    target = min(len(items) for items in by_area.values())
    balanced_pending = []
    balanced_resumen = []
    for area_id in sorted(by_area):
        for item in by_area[area_id][:target]:
            old_index = item['summary_index']
            item['summary_index'] = len(balanced_resumen)
            balanced_resumen.append(resumen[old_index])
            balanced_pending.append(item)
    pending, resumen = balanced_pending, balanced_resumen

    if persist_to_db:
        persist_cases(pending, resumen)
    else:
        for index, item in enumerate(resumen, start=1):
            item['id_usuario'] = f'SIM-C{index}'
    return resumen


resumen_combo = run_combined_cross_perturbation(persist_to_db=True)
df_combo = pl.DataFrame(resumen_combo)
print(f'Casos Caso C insertados: {df_combo.height}')
print(df_combo.group_by('id_area').len().sort('id_area'))
print(f'Rango: [{df_combo["porcentaje_coincidencia"].min():.4f}, {df_combo["porcentaje_coincidencia"].max():.4f}]')
display(df_combo.head(8))

Casos Caso C insertados: 726
shape: (3, 2)
┌─────────┬─────┐
│ id_area ┆ len │
│ ---     ┆ --- │
│ i64     ┆ u32 │
╞═════════╪═════╡
│ 1       ┆ 242 │
│ 2       ┆ 242 │
│ 3       ┆ 242 │
└─────────┴─────┘
Rango: [0.0000, 0.8000]


id_usuario,uuid_usuario,variacion,id_area,id_carrera,carrera_evaluada,total_requeridas,habilidades_perfil_ruidoso,habilidades_comunes,suma_cumplimiento,porcentaje_coincidencia
i64,str,i64,i64,i64,str,i64,i64,i64,f64,f64
496,"""COMBO-5a8b0e9e-b06c-4885-ba7e-…",1,1,1,"""Ingeniería Aeronáutica""",7,6,6,5.22,0.7457
497,"""COMBO-0063d06f-1ab7-43e3-9baf-…",1,1,2,"""Ingeniería Ambiental""",6,6,1,1.0,0.1667
498,"""COMBO-98416440-788e-4f04-be47-…",1,1,3,"""Ingeniería Biomédica""",6,6,1,1.0,0.1667
499,"""COMBO-aa0ef876-88cd-456c-af33-…",1,1,4,"""Ingeniería Biónica""",6,6,0,0.0,0.0
500,"""COMBO-e25bbf37-e2fb-4e4c-b25f-…",1,1,5,"""Ingeniería Bioquímica""",6,6,0,0.0,0.0
501,"""COMBO-acc0f93d-e197-4c7d-bd87-…",1,1,6,"""Ingeniería Biotecnológica""",5,6,0,0.0,0.0
502,"""COMBO-0dcff279-e307-47d8-a325-…",1,1,7,"""Ingeniería Civil""",5,6,1,1.0,0.2
503,"""COMBO-1ea6e9e1-6f0d-4146-bc21-…",1,1,8,"""Ingeniería Eléctrica""",5,6,0,0.0,0.0


### Celda 5: Distribución de Deciles para Caso C

In [6]:
dist_c = df_combo.with_columns(
    ((pl.col('porcentaje_coincidencia') * 10).floor() / 10.0).alias('decil')
).group_by('decil').len().sort('decil')

print('=== DISTRIBUCIÓN PORCENTUAL CASO C ===')
total_c = df_combo.height
for row in dist_c.iter_rows(named=True):
    pct = (row['len'] / total_c) * 100
    barra = '█' * int(pct // 2)
    print(f"{row['decil']:.1f} - {row['decil']+0.1:.1f}: {row['len']:4d} casos ({pct:5.1f}%) | {barra}")

=== DISTRIBUCIÓN PORCENTUAL CASO C ===
0.0 - 0.1:  485 casos ( 66.8%) | █████████████████████████████████
0.1 - 0.2:   40 casos (  5.5%) | ██
0.2 - 0.3:  152 casos ( 20.9%) | ██████████
0.3 - 0.4:    1 casos (  0.1%) | 
0.5 - 0.6:   12 casos (  1.7%) | 
0.6 - 0.7:   11 casos (  1.5%) | 
0.7 - 0.8:   21 casos (  2.9%) | █
0.8 - 0.9:    4 casos (  0.6%) | 


In [7]:
# Verificacion de cobertura de las tres areas en Caso C.
expected_area_ids = {area['id_area'] for area in get_all_areas()}
if len(expected_area_ids) != 3:
    raise ValueError(
        f'Se esperaban 3 areas configuradas y se encontraron {len(expected_area_ids)}.'
    )

actual_area_ids = set(df_combo['id_area'].unique().to_list())
missing_area_ids = expected_area_ids - actual_area_ids
if missing_area_ids:
    raise ValueError(
        f'Caso C no genero usuarios en las areas: {sorted(missing_area_ids)}'
    )

print('Cobertura Caso C por area:')
print(df_combo.group_by('id_area').len().sort('id_area'))

Cobertura Caso C por area:
shape: (3, 2)
┌─────────┬─────┐
│ id_area ┆ len │
│ ---     ┆ --- │
│ i64     ┆ u32 │
╞═════════╪═════╡
│ 1       ┆ 242 │
│ 2       ┆ 242 │
│ 3       ┆ 242 │
└─────────┴─────┘
